# NB26 · Type Hints

`advanced/` · 20 questions · the last one is a story project

Type hints say what kind of value a name should hold, and Python never
checks them while the program runs. Tools like mypy catch mistakes before
you run anything. But the hints are still real objects at runtime, so you
can read them with `typing.get_type_hints`, `get_origin` and `get_args`,
and use them to validate real data. Every task here prints something that
depends on that runtime behaviour: reading hints, `isinstance` checks
against a `@runtime_checkable` Protocol, generics, and small validators.

## You'll need

- NB25: metaprogramming
- NB21: dataclasses

## You'll practise

- Annotating variables, parameters and return values
- Built-in generics (list[int]); X | None; Callable; collections.abc types
- Generics: TypeVar, and the PEP 695 syntax def f[T](x: T) and class Box[T]
- type aliases with the type statement (3.12)
- Protocol, TypedDict, Literal, Final, ClassVar, Self, overload, NewType, cast
- Hints at runtime: get_type_hints, deferred annotations in 3.14
- What mypy/pyright catch that tests don't
- Newer typing: @override (3.12), TypeIs and ReadOnly (3.13), type-parameter defaults (3.13)

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Questions get longer as you go: one-liners first, a small story project last. Every solution only uses what this notebook and the ones before it teach.

---

## Warm-up · one line each

### Q01 · Output-match

`price_with_tax` is already annotated. Print the dictionary that
`typing.get_type_hints` returns for it.

**Expected output:**

```text
{'price': <class 'float'>, 'rate': <class 'float'>, 'rounded': <class 'bool'>, 'return': <class 'float'>}
```

<details><summary><b>Solution</b></summary>

```python
import typing

print(typing.get_type_hints(price_with_tax))
```

</details>

In [ ]:
def price_with_tax(price: float, rate: float, rounded: bool = True) -> float:
    total = price * (1 + rate)
    return round(total, 2) if rounded else total

# your code here


### Q02 · Puzzle

The hint on `double` says `int`, but the shop's form sent text. Call it
with `"ab"` and print the result, then print whether `"ab"` is an
instance of `int`.

**Expected output:**

```text
abab
False
```

<details><summary><b>Solution</b></summary>

```python
print(double("ab"))
print(isinstance("ab", int))
```

</details>

In [ ]:
def double(n: int) -> int:
    return n * 2

# your code here


### Q03 · Output-match

A table of scores per student is typed as `dict[str, list[int]]`. Print its origin
(the plain class) and then its arguments, each on its own line.

**Expected output:**

```text
<class 'dict'>
(<class 'str'>, list[int])
```

<details><summary><b>Solution</b></summary>

```python
print(get_origin(scores_hint))
print(get_args(scores_hint))
```

</details>

In [ ]:
from typing import get_origin, get_args

scores_hint = dict[str, list[int]]

# your code here


### Q04 · Puzzle

Someone wants to check that `grades` is a `list[int]` with
`isinstance(grades, list[int])`. Try it, catch the `TypeError` and print
`cannot check`. Then check it properly: print `True` if `grades` is a list
and every item is an `int`.

**Expected output:**

```text
cannot check
True
```

<details><summary><b>Solution</b></summary>

```python
try:
    isinstance(grades, list[int])
except TypeError:
    print("cannot check")
print(isinstance(grades, list) and all(isinstance(g, int) for g in grades))
```

</details>

In [ ]:
grades = [90, 85, 77]

# your code here


### Q05 · Puzzle

A user's nickname is typed `str | None`. Print the names of the two types
inside that hint, in the order they were written, as a list of strings.

**Expected output:**

```text
['str', 'NoneType']
```

<details><summary><b>Solution</b></summary>

```python
print([t.__name__ for t in get_args(nickname_hint)])
```

</details>

In [ ]:
from typing import get_args

nickname_hint = str | None

# your code here


---

## Combine · 2 to 4 lines

### Q06 · Real-life

A drawing app accepts "anything with an `.area()` method". Make a
`HasArea` Protocol that can be used with `isinstance`, then print, one per
line, whether `circle`, `square` and `rock` are instances of it.

**Expected output:**

```text
True
True
False
```

<details><summary><b>Solution</b></summary>

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class HasArea(Protocol):
    def area(self) -> float: ...

for shape in (circle, square, rock):
    print(isinstance(shape, HasArea))
```

</details>

In [ ]:
class Circle:
    def __init__(self, r):
        self.r = r
    def area(self):
        return 3.14 * self.r ** 2

class Square:
    def __init__(self, side):
        self.side = side
    def area(self):
        return self.side ** 2

class Rock:
    weight = 12

circle, square, rock = Circle(1), Square(2), Rock()

# your code here


### Q07 · Real-life

A signup form record must match the `User` TypedDict. Print the required
keys that are missing from `record` and then the keys that `User` doesn't
know about, each as a sorted list.

**Expected output:**

```text
['email']
['nickname']
```

<details><summary><b>Solution</b></summary>

```python
known = set(User.__annotations__)
print(sorted(User.__required_keys__ - set(record)))
print(sorted(set(record) - known))
```

</details>

In [ ]:
from typing import TypedDict

class User(TypedDict):
    name: str
    email: str
    age: int

record = {"name": "Mia", "age": 31, "nickname": "mimi"}

# your code here


### Q08 · Real-life

An order's status must be one of the values allowed by the `Status`
Literal. Print the order ids whose status is not allowed, as a list.

**Expected output:**

```text
[2, 4]
```

<details><summary><b>Solution</b></summary>

```python
allowed = get_args(Status)
print([order_id for order_id, status in orders if status not in allowed])
```

</details>

In [ ]:
from typing import Literal, get_args

Status = Literal["new", "paid", "shipped"]

orders = [(1, "paid"), (2, "shiped"), (3, "new"), (4, "PAID")]

# your code here


### Q09 · Build

Write a generic `Stack[T]` class with the new syntax, with `push(item)`
and `pop()` methods (store items in a list). Push `1`, `2`, `3`, pop once
and print the popped value, then print `Stack.__type_params__`.

**Expected output:**

```text
3
(T,)
```

<details><summary><b>Solution</b></summary>

```python
class Stack[T]:
    def __init__(self):
        self.items: list[T] = []

    def push(self, item: T) -> None:
        self.items.append(item)

    def pop(self) -> T:
        return self.items.pop()

stack = Stack()
for n in (1, 2, 3):
    stack.push(n)
print(stack.pop())
print(Stack.__type_params__)
```

</details>

In [ ]:
# your code here


### Q10 · Build

Make a type alias `Point` for a `tuple[float, float]` using the `type`
statement. Write `distance(a, b)` annotated with it (straight-line
distance), print `distance` between `(0, 0)` and `(3, 4)`, then print
`Point.__name__` and `Point.__value__`.

**Expected output:**

```text
5.0
Point
tuple[float, float]
```

<details><summary><b>Solution</b></summary>

```python
type Point = tuple[float, float]

def distance(a: Point, b: Point) -> float:
    return ((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2) ** 0.5

print(distance((0, 0), (3, 4)))
print(Point.__name__)
print(Point.__value__)
```

</details>

In [ ]:
# your code here


---

## Small real tasks · 5 to 8 lines

### Q11 · Real-life

A `Product` dataclass has a `ClassVar` counter and three real fields.
Create two products, then print the counter and the field names that the
dataclass machinery sees (the `ClassVar` is not a field).

**Expected output:**

```text
2
['sku', 'price', 'stock']
```

<details><summary><b>Solution</b></summary>

```python
Product("A1", 2.5)
Product("B2", 9.0, stock=4)
print(Product.created)
print([f.name for f in fields(Product)])
```

</details>

In [ ]:
from dataclasses import dataclass, fields
from typing import ClassVar

@dataclass
class Product:
    created: ClassVar[int] = 0
    sku: str
    price: float
    stock: int = 0

    def __post_init__(self):
        Product.created += 1

# your code here


### Q12 · Puzzle

`UserId` and `OrderId` are `NewType`s over `int`, so type checkers keep
them apart. At runtime they are just the plain value. Create `UserId(7)`
and `OrderId(7)`, then print whether they are equal, the type name of
`UserId(7)`, and `UserId.__supertype__`.

**Expected output:**

```text
True
int
<class 'int'>
```

<details><summary><b>Solution</b></summary>

```python
user, order = UserId(7), OrderId(7)
print(user == order)
print(type(user).__name__)
print(UserId.__supertype__)
```

</details>

In [ ]:
from typing import NewType

UserId = NewType("UserId", int)
OrderId = NewType("OrderId", int)

# your code here


### Q13 · Build

Write `parse(value)` with two `@overload` stubs (a `str` gives back an
`int`, a `list[str]` gives back a `list[int]`) and a real body that does
the dispatch by hand with `isinstance`. Print `parse("42")`,
`parse(["1", "2"])` and the number of overloads that
`typing.get_overloads(parse)` reports.

**Expected output:**

```text
42
[1, 2]
2
```

<details><summary><b>Solution</b></summary>

```python
import typing
from typing import overload

@overload
def parse(value: str) -> int: ...
@overload
def parse(value: list[str]) -> list[int]: ...
def parse(value):
    if isinstance(value, str):
        return int(value)
    return [int(v) for v in value]

print(parse("42"))
print(parse(["1", "2"]))
print(len(typing.get_overloads(parse)))
```

</details>

In [ ]:
# your code here


### Q14 · Real-life

`Settings` lists the type of each config value. For every key in `config`
print `<key>: ok` if the value matches the hint, otherwise
`<key>: expected <type name>`. Use `typing.get_type_hints`, and remember
that `True` is also an `int`: a `port` of `True` must be rejected.

**Expected output:**

```text
host: ok
port: expected int
debug: ok
ratio: expected float
```

<details><summary><b>Solution</b></summary>

```python
import typing

hints = typing.get_type_hints(Settings)
for key, value in config.items():
    expected = hints[key]
    good = isinstance(value, expected)
    if expected is int and isinstance(value, bool):
        good = False
    print(f"{key}: ok" if good else f"{key}: expected {expected.__name__}")
```

</details>

In [ ]:
class Settings:
    host: str
    port: int
    debug: bool
    ratio: float

config = {"host": "db.local", "port": True, "debug": False, "ratio": "0.5"}

# your code here


### Q15 · Build

A shape-drawing helper `total_area` accepts a list of anything that has
an `.area()` method. Write a `HasArea` Protocol, then write `total_area`
as a generic function whose type parameter `T` is bounded by `HasArea`
(new syntax). Print the total area of `shapes`, then the name of the bound
class: `total_area.__type_params__[0].__bound__.__name__`.

**Expected output:**

```text
27
HasArea
```

<details><summary><b>Solution</b></summary>

```python
from typing import Protocol

class HasArea(Protocol):
    def area(self) -> float: ...

def total_area[T: HasArea](items: list[T]) -> float:
    return sum(item.area() for item in items)

print(total_area(shapes))
print(total_area.__type_params__[0].__bound__.__name__)
```

</details>

In [ ]:
class Rect:
    def __init__(self, w, h):
        self.w, self.h = w, h
    def area(self):
        return self.w * self.h

class Tile:
    def area(self):
        return 1

shapes = [Rect(2, 3), Tile(), Rect(4, 5)]

# your code here


---

## Puzzles & builds · 8 to 15 lines

### Q16 · Puzzle

A report builder has a `Report` base class whose methods chain by
returning `Self`. `HtmlReport` replaces `render` and marks it with
`@override` so a checker would catch a typo. Build both: `add(line)`
stores a line and returns `self`; `render()` joins lines with `|` in
`Report`, and wraps `<p>..</p>` around each line in `HtmlReport`. Print
the HTML render of a chain adding `a` then `b`, and then
`HtmlReport.render.__override__` and `hasattr(Report.render, "__override__")`.

**Expected output:**

```text
<p>a</p><p>b</p>
True
False
```

<details><summary><b>Solution</b></summary>

```python
from typing import Self, override

class Report:
    def __init__(self):
        self.lines: list[str] = []

    def add(self, line: str) -> Self:
        self.lines.append(line)
        return self

    def render(self) -> str:
        return "|".join(self.lines)

class HtmlReport(Report):
    @override
    def render(self) -> str:
        return "".join(f"<p>{line}</p>" for line in self.lines)

print(HtmlReport().add("a").add("b").render())
print(HtmlReport.render.__override__)
print(hasattr(Report.render, "__override__"))
```

</details>

In [ ]:
# your code here


### Q17 · Build

Write a decorator `enforce` that makes a function's hints real: before the
call it reads `typing.get_type_hints` and `inspect.signature(...).bind`,
and raises `TypeError("<name> must be <type name>")` for the first
argument whose class doesn't match its hint (parameters without hints are
skipped). Decorate `book_seats(show: str, seats: int)` to return
`"<show>: <seats> seats"`. Print the result of a good call, then call it
with `seats="3"` and print the error message.

**Expected output:**

```text
Hamlet: 2 seats
seats must be int
```

<details><summary><b>Solution</b></summary>

```python
import functools
import inspect
import typing

def enforce(func):
    hints = typing.get_type_hints(func)
    signature = inspect.signature(func)

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        bound = signature.bind(*args, **kwargs)
        for name, value in bound.arguments.items():
            expected = hints.get(name)
            if expected is not None and not isinstance(value, expected):
                raise TypeError(f"{name} must be {expected.__name__}")
        return func(*args, **kwargs)
    return wrapper

@enforce
def book_seats(show: str, seats: int):
    return f"{show}: {seats} seats"

print(book_seats("Hamlet", 2))
try:
    book_seats("Hamlet", seats="3")
except TypeError as error:
    print(error)
```

</details>

In [ ]:
# your code here


### Q18 · Build

Write `load(cls, data)` that builds an instance of a dataclass from a
plain dict using the dataclass's hints. For each field: a missing key with
no default raises `ValueError("missing <field>")`; a value of the wrong
class raises `ValueError("<field> must be <type name>")`; otherwise it is
used. Fields hinted `int | None` accept `None`. Load the three dicts in
`rows` into `Member` and print each result, or the error message.

**Expected output:**

```text
Member(name='Ana', age=30, team='red')
missing age
age must be int
```

<details><summary><b>Solution</b></summary>

```python
import dataclasses
import typing

def load(cls, data):
    hints = typing.get_type_hints(cls)
    values = {}
    for field in dataclasses.fields(cls):
        if field.name not in data:
            if field.default is dataclasses.MISSING:
                raise ValueError(f"missing {field.name}")
            continue
        value = data[field.name]
        if not isinstance(value, hints[field.name]):
            kinds = typing.get_args(hints[field.name]) or (hints[field.name],)
            raise ValueError(f"{field.name} must be {kinds[0].__name__}")
        values[field.name] = value
    return cls(**values)

for row in rows:
    try:
        print(load(Member, row))
    except ValueError as error:
        print(error)
```

</details>

In [ ]:
from dataclasses import dataclass

@dataclass
class Member:
    name: str
    age: int
    team: str | None = None

rows = [
    {"name": "Ana", "age": 30, "team": "red"},
    {"name": "Bo"},
    {"name": "Cy", "age": "41"},
]

# your code here


### Q19 · Puzzle

A function's annotations hide the type tree. Write `describe(hint)` that
prints the hint as nested text: for a plain class its `__name__`; for a
generic `origin[arg1, arg2]` using the origin's name and the described
args (recursively), separated by `, `; for a union (`X | Y`) the described
parts joined with ` or `. Print the description of each hint in `hints`.

**Expected output:**

```text
int
list[str]
dict[str, list[int]]
int or NoneType
list[int or str]
```

<details><summary><b>Solution</b></summary>

```python
def describe(hint):
    origin = get_origin(hint)
    args = get_args(hint)
    if origin is None:
        return hint.__name__
    if origin is types.UnionType:
        return " or ".join(describe(a) for a in args)
    return f"{origin.__name__}[{', '.join(describe(a) for a in args)}]"

for hint in hints:
    print(describe(hint))
```

</details>

In [ ]:
from typing import get_args, get_origin
import types

hints = [int, list[str], dict[str, list[int]], int | None, list[int | str]]

# your code here


---

## Q20 · Story project: Typed Repository Layer

You are building a small storage layer for a shop. The record type is a
dataclass, storage backends are described by a `Protocol`, and a generic
`Repository[T]` checks every record against the dataclass's hints before
saving it. Each step builds on the previous ones.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
from dataclasses import dataclass

@dataclass
class Item:
    sku: str
    name: str
    price: float
    stock: int = 0


### Step 1

Write `check(cls, data)` that returns a list of problem strings for a dict `data` against the dataclass `cls`, using `typing.get_type_hints`: `missing <field>` for a required field absent, `<field> must be <type name>` for a wrong class (an `int` is fine where `float` is expected). Print the problems for a good dict, and for `{"sku": "A1", "price": "cheap"}`.

**Expected output:**

```text
[]
['missing name', 'price must be float']
```

<details><summary><b>Solution</b></summary>

```python
import dataclasses
import typing

def check(cls, data):
    hints = typing.get_type_hints(cls)
    problems = []
    for field in dataclasses.fields(cls):
        if field.name not in data:
            if field.default is dataclasses.MISSING:
                problems.append(f"missing {field.name}")
            continue
        expected = hints[field.name]
        if expected is float:
            expected = (int, float)
        if not isinstance(data[field.name], expected):
            problems.append(f"{field.name} must be {hints[field.name].__name__}")
    return problems

print(check(Item, {"sku": "A1", "name": "Pen", "price": 2, "stock": 5}))
print(check(Item, {"sku": "A1", "price": "cheap"}))
```

</details>

In [ ]:
# your code here


### Step 2

Write a `Storage` Protocol (runtime checkable) with `save(key, value)` and `load(key)`, and a `MemoryStorage` class backed by a dict (`load` returns `None` for an unknown key). Print `isinstance(MemoryStorage(), Storage)` and `isinstance(object(), Storage)`.

**Expected output:**

```text
True
False
```

<details><summary><b>Solution</b></summary>

```python
from typing import Protocol, runtime_checkable

@runtime_checkable
class Storage(Protocol):
    def save(self, key: str, value: dict) -> None: ...
    def load(self, key: str) -> dict | None: ...

class MemoryStorage:
    def __init__(self):
        self.data = {}

    def save(self, key, value):
        self.data[key] = value

    def load(self, key):
        return self.data.get(key)

print(isinstance(MemoryStorage(), Storage))
print(isinstance(object(), Storage))
```

</details>

In [ ]:
# your code here


### Step 3

Write `class Repository[T]` taking a record class and a storage. `add(data)` raises `ValueError` with the problems joined by `; ` if `check` finds any, otherwise saves the dict under its `sku` and returns the built `T`. `get(sku)` returns a `T` or `None`. Add the good item, try the bad one and print the error, then print `get("A1")` and `get("Z9")`.

**Expected output:**

```text
missing name; price must be float
Item(sku='A1', name='Pen', price=2, stock=5)
None
```

<details><summary><b>Solution</b></summary>

```python
class Repository[T]:
    def __init__(self, cls: type[T], storage: Storage):
        self.cls = cls
        self.storage = storage

    def add(self, data: dict) -> T:
        problems = check(self.cls, data)
        if problems:
            raise ValueError("; ".join(problems))
        self.storage.save(data["sku"], data)
        return self.cls(**data)

    def get(self, sku: str) -> T | None:
        data = self.storage.load(sku)
        return self.cls(**data) if data is not None else None

repo = Repository(Item, MemoryStorage())
repo.add({"sku": "A1", "name": "Pen", "price": 2, "stock": 5})
try:
    repo.add({"sku": "B2", "price": "cheap"})
except ValueError as error:
    print(error)
print(repo.get("A1"))
print(repo.get("Z9"))
```

</details>

In [ ]:
# your code here


### Step 4

Write `JsonStorage(path)` that keeps everything in one JSON file (read the whole file on each call, write it back on `save`; a missing file counts as empty). Inside a temporary directory make a `Repository(Item, JsonStorage(...))`, add two items, then create a second repository on the same file and print `get` for one of them, to prove the data persisted. Also print `isinstance` of the storage against `Storage`.

**Expected output:**

```text
Item(sku='D4', name='Pad', price=3, stock=2)
True
```

<details><summary><b>Solution</b></summary>

```python
import json
import tempfile
from pathlib import Path

class JsonStorage:
    def __init__(self, path):
        self.path = Path(path)

    def _read(self):
        if not self.path.exists():
            return {}
        return json.loads(self.path.read_text())

    def save(self, key, value):
        data = self._read()
        data[key] = value
        self.path.write_text(json.dumps(data))

    def load(self, key):
        return self._read().get(key)

with tempfile.TemporaryDirectory() as folder:
    storage = JsonStorage(Path(folder) / "items.json")
    first = Repository(Item, storage)
    first.add({"sku": "C3", "name": "Ink", "price": 7.5})
    first.add({"sku": "D4", "name": "Pad", "price": 3, "stock": 2})
    second = Repository(Item, JsonStorage(storage.path))
    print(second.get("D4"))
    print(isinstance(storage, Storage))
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB27 · Concurrency Asyncio** in `advanced/`.